# AttGAN — Facial Attribute Editing with GANs
**Dataset:** CelebA (torchvision) · **Framework:** PyTorch · **Platform:** Google Colab
> He et al. (2019) — *AttGAN: Facial Attribute Editing by Only Changing What You Want*

---
## Before you start
1. **Set runtime to GPU** → `Runtime → Change runtime type → T4 GPU → Save`
2. Run all cells top to bottom with `Shift+Enter`
3. CelebA (~1.4 GB) downloads automatically in Cell 4 on first run

---
## How experiments work
This notebook runs **one experiment at a time**. Change `EXPERIMENT` in **Cell 3**,
then run the full notebook. Do this three times to cover all configurations.

| Experiment | λ_rec | λ_cls_G | What changes |
|---|---|---|---|
| `exp1_baseline` | 100 | 1 | Paper defaults — reference run |
| `exp2_high_rec` | 200 | 1 | Stronger identity preservation |
| `exp3_strong_attr` | 50 | 5 | Sharper attribute edits |

After all three runs, execute **Cell 13 — Export** to generate comparison charts.

---
## Cell 1 — Clone repo & install dependencies

In [ ]:
# If the repo was already cloned this session, skip the clone line
!git clone https://github.com/YOUR_USERNAME/GAN_Project_DL.git
%cd GAN_Project_DL
!pip install -q -r requirements.txt

---
## Cell 2 — Verify GPU & environment

In [ ]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('No GPU found. Go to Runtime > Change Runtime Type > T4 GPU')

---
## Cell 3 — Select experiment & load config
**Change `EXPERIMENT` here before running the rest of the notebook.**
Each value maps to a different set of loss weights in `experiments/`.
Run the full notebook once per experiment, then use Cell 13 to compare.

In [ ]:
import importlib, torch

# ── CHANGE THIS to switch experiments ───────────────────────────────
EXPERIMENT = 'exp1_baseline'
# Options:
#   'exp1_baseline'    -> lambda_rec=100, lambda_cls_g=1   (paper defaults)
#   'exp2_high_rec'    -> lambda_rec=200, lambda_cls_g=1   (stronger identity)
#   'exp3_strong_attr' -> lambda_rec=50,  lambda_cls_g=5   (sharper edits)
# ────────────────────────────────────────────────────────────────────

_EXP_MAP = {
    'exp1_baseline':    ('experiments.exp1_baseline', 'Exp1Config'),
    'exp2_high_rec':    ('experiments.exp2_high_rec', 'Exp2Config'),
    'exp3_strong_attr': ('experiments.exp3_low_rec',  'Exp3Config'),
}
mod_path, cls_name = _EXP_MAP[EXPERIMENT]
cfg = getattr(importlib.import_module(mod_path), cls_name)()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Experiment   : {cfg.EXPERIMENT_NAME}')
print(f'Device       : {device}')
print(f'lambda_rec   : {cfg.LAMBDA_REC}')
print(f'lambda_cls_D : {cfg.LAMBDA_CLS_D}')
print(f'lambda_cls_G : {cfg.LAMBDA_CLS_G}')
print(f'Epochs       : {cfg.N_EPOCHS}')
print(f'Batch size   : {cfg.BATCH_SIZE}')
print(f'Results dir  : {cfg.RESULTS_DIR}')
if hasattr(cfg, 'DESCRIPTION'):
    print(f'Description  : {cfg.DESCRIPTION}')

---
## Cell 4 — Download & load CelebA
torchvision downloads CelebA automatically (~1.4 GB) on the first run.
Subsequent runs load from the cached copy in `data/`.

**Splits used (torchvision defaults):**

| Split | Images |
|---|---|
| train | 162,770 |
| valid | 19,867 |
| test  | 19,962 |

> If you get `FileURLRetrievalError: Too many users`, the Google Drive quota is exceeded.
> Run the **fallback cell at the bottom** of this notebook instead.

In [ ]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

# Sanity check
imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}   (B, C, H, W)')
print(f'Attr  batch : {attrs.shape}')
print(f'Attr values : {set(attrs.unique().tolist())}  (should be {{-1.0, 1.0}})')
print(f'Pixel range : [{imgs.min():.2f}, {imgs.max():.2f}]  (should be ~[-1, 1])')

---
## Cell 5 — Build models
```
image (3x128x128)
  |--> Encoder  (5x Conv stride-2)     --> z  (512 x 4 x 4)
                                              |
                         target_attrs --------+  (tiled spatially, concat)
                                              |
                                         Generator (5x ConvTranspose) --> fake (3x128x128)
                                                                               |
                                         Discriminator <-----------------------+
                                           |--> adv_head  (LSGAN real/fake)
                                           |--> cls_head  (13 attribute logits, BCE)
```

In [ ]:
from src.models import build_models

enc, gen, dis = build_models(cfg, device)

---
## Cell 6 — Train

| Loss | Function | Weight | Purpose |
|---|---|---|---|
| L_adv | MSELoss (LSGAN) | — | Realism |
| L_cls | BCEWithLogitsLoss | lambda_cls_D / lambda_cls_G | Correct attributes |
| L_rec | L1Loss | lambda_rec | Identity preservation |

**Total G loss:** `L_adv + lambda_cls_G * L_cls + lambda_rec * L_rec`

Samples and checkpoints are saved to `results/<experiment>/` every 5 epochs.

In [ ]:
from src.trainer import Trainer

trainer = Trainer(enc, gen, dis, train_loader, test_loader, cfg, device)

# To resume from a saved checkpoint uncomment:
# resume = f'checkpoints/{cfg.EXPERIMENT_NAME}/ckpt_epoch010.pt'
# g_losses, d_losses = trainer.train(resume_path=resume)

g_losses, d_losses = trainer.train()

---
## Cell 7 — Loss curves

In [ ]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

---
## Cell 8 — Attribute manipulation demo
Each row = one test image.  
Each column = one attribute toggled independently from its current value.  
The model should only change the requested attribute — everything else stays the same.

In [ ]:
from src.utils import attribute_demo
attribute_demo(enc, gen, test_loader, cfg, n_imgs=4)

---
## Cell 9 — Attribute accuracy & reconstruction quality

In [ ]:
from src.utils import evaluate_attribute_accuracy, evaluate_reconstruction

acc = evaluate_attribute_accuracy(enc, gen, dis, test_loader, cfg)
rec = evaluate_reconstruction(enc, gen, test_loader, cfg)

print(f'Attribute accuracy (avg) : {acc:.1f}%')
print(f'Reconstruction L1        : {rec:.4f}')

---
## Cell 10 — FID & DACID metrics
**FID** (Frechet Inception Distance): standard GAN quality metric.
Frechet distance between Inception-v3 feature distributions of real vs generated images. Lower = better.

**DACID** (Dany Aissa & Clara's Image Distance): custom lightweight metric.
L2 distance between the *mean* Inception feature vectors — faster than FID, easy to compare across experiments. Lower = better.

> Takes ~5 extra minutes on T4. Set `cfg.COMPUTE_METRICS = False` to skip.

In [ ]:
# Uncomment to skip metrics and save time:
# cfg.COMPUTE_METRICS = False

if cfg.COMPUTE_METRICS:
    from src.metrics import compute_metrics
    scores = compute_metrics(enc, gen, test_loader, cfg, device)
    print(f"FID   : {scores['fid']}   (lower = better)")
    print(f"DACID : {scores['dacid']}   (lower = better)")
else:
    scores = {}
    print('Metrics skipped (COMPUTE_METRICS = False)')

---
## Cell 11 — Save metrics.json
Saves all scores and loss history to `results/<experiment>/metrics.json`.  
**Run this at the end of every experiment** — Cell 13 reads these files to build the comparison.

In [ ]:
import json

payload = {
    'experiment':   cfg.EXPERIMENT_NAME,
    'model':        'AttGAN',
    'lambda_rec':   cfg.LAMBDA_REC,
    'lambda_cls_d': cfg.LAMBDA_CLS_D,
    'lambda_cls_g': cfg.LAMBDA_CLS_G,
    'fid':          scores.get('fid'),
    'dacid':        scores.get('dacid'),
    'attr_acc':     round(float(acc), 2) if 'acc' in dir() else None,
    'rec_l1':       round(float(rec), 4) if 'rec' in dir() else None,
    'g_losses':     g_losses,
    'd_losses':     d_losses,
}

out = cfg.RESULTS_DIR / 'metrics.json'
with open(out, 'w') as f:
    json.dump(payload, f, indent=2)

print(f'Saved -> {out}')
print(f'\nSummary for {cfg.EXPERIMENT_NAME}:')
for k in ['lambda_rec', 'lambda_cls_g', 'fid', 'dacid', 'attr_acc', 'rec_l1']:
    print(f'  {k:<16}: {payload[k]}')

---
## Cell 12 — Download results to your computer
Downloads the results folder for this experiment so you keep your outputs
even after the Colab session ends.

In [ ]:
import shutil
from google.colab import files

zip_name = f'{cfg.EXPERIMENT_NAME}_results'
shutil.make_archive(zip_name, 'zip', cfg.RESULTS_DIR)
files.download(f'{zip_name}.zip')
print(f'Downloaded {zip_name}.zip')

---
## Cell 13 — Export & compare all experiments
**Run after completing ALL THREE experiments.**  
Reads every `results/exp*/metrics.json` and produces:
- `results/comparison_table.csv`
- `results/comparison_metrics.png` (FID & DACID bar charts)
- `results/comparison_losses.png` (overlaid loss curves)

Missing experiments are skipped automatically.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('export_results', 'export_results.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

---
## Fallback — CelebA download quota exceeded
If Cell 4 raises `FileURLRetrievalError: Too many users have viewed or downloaded this file`,
the Google Drive quota is temporarily exceeded. Use one of these alternatives:

**Option A — Kaggle API (recommended)**

In [ ]:
# Option A — Kaggle API
# 1. Get your Kaggle API token: kaggle.com -> Account -> Create New Token -> downloads kaggle.json
# 2. Upload kaggle.json to Colab (Files panel on the left), then run:

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d jessicali9530/celeba-dataset -p data/
# !unzip -q data/celeba-dataset.zip -d data/celeba_kaggle

# Then point cfg to the extracted folder:
# from src.dataset import get_loaders as _get
# from pathlib import Path
# # torchvision expects its own folder layout — use manual loader instead:
# # (see dataset.py for the torchvision folder structure)

print('Uncomment the lines above and run after uploading kaggle.json')

In [ ]:
# Option B — Mount Google Drive with a pre-downloaded copy
# If you have CelebA already in your Drive (from a previous session):

# from google.colab import drive
# drive.mount('/content/drive')

# import os
# # Symlink Drive copy into the expected torchvision path
# os.makedirs('data', exist_ok=True)
# !ln -sfn /content/drive/MyDrive/celeba data/celeba

# Then re-run Cell 4 with download=False:
# from src.dataset import CelebAAttrDataset
# # (torchvision will find the files in data/celeba/)

print('Uncomment the lines above after mounting your Drive')